# Density

Past a certain count, colouring individual points stops showing anything. `add_heatmap`
sums weights into cells and colours the sums, in two flavours: screen-space **blobs**,
and real **cell polygons** (H3 hexagons or geohash rectangles) filled through the ramp.

The cell kernels and `hexbin` need `h3` (`pip install h3`); blobs need nothing extra.

In [ ]:
import numpy as np
import pandas as pd
import swiftmap
from swiftmap import Map

rng = np.random.default_rng(11)
# Three clusters of very different density, plus a scatter of stragglers.
parts = [
    (4000, 36.05, -5.45, 0.010),
    (1500, 36.18, -5.28, 0.014),
    ( 400, 35.92, -5.62, 0.020),
    ( 600, 36.05, -5.45, 0.120),
]
df = pd.concat([
    pd.DataFrame({"lat": clat + rng.normal(0, s, n),
                  "lon": clon + rng.normal(0, s * 1.6, n),
                  "weight": rng.gamma(2, 2, n)})
    for n, clat, clon, s in parts
], ignore_index=True)
len(df)

## Blobs

A Gaussian kernel sized in **pixels**, accumulated on the GPU. Because the kernel is
screen-space, the field recomputes with every view.

In [ ]:
m = Map()
m.add_heatmap(df, name="Density")
m.configure_legend(show=True)
m

Zoom into one cluster: the colours re-stretch to what is on screen, so a quiet region
shows its own structure instead of staying uniformly dark. That is also why the legend
reads low to high without numbers — a number on a view-relative scale would change with
every pan.

Pin the scale when you want the colours to mean one fixed thing:

In [ ]:
m2 = Map()
m2.add_heatmap(df, name="Pinned", max_intensity=25, auto_normalize=False)
m2

## Real cells

`cells="h3"` bins in Python at add time and fills the actual hexagons. `weight_col` sums
a column instead of counting rows.

In [ ]:
m3 = Map()
m3.add_heatmap(df, cells="h3", resolution=7, weight_col="weight", name="Hex density")
m3.configure_legend(show=True)
m3

Geohash rectangles are the other cell family, through the same path. `base` is required —
a hash cannot state its own base, so swiftmap never assumes one.

In [ ]:
m4 = Map()
m4.add_heatmap(df, cells="geohash", length=5, base=32, name="Geohash density")
m4

## From a layer already on the map

Naming an existing point layer derives the heat from its buffer — a config, not a second
upload — and reads `weight_col` from that layer's own properties.

In [ ]:
m5 = Map()
m5.add_circle_markers(df, name="Sites", radius=3, color="#666")
m5.add_heatmap("Sites", weight_col="weight", name="Heat", layer_group="Density")
m5

## When you want numbers: bin it yourself

`add_heatmap` is the only place swiftmap aggregates. Anything you want to click, label,
or read a value off is better binned as data and drawn as polygons — which is what
`swiftmap.hexbin` is for: data in, data out.

In [ ]:
cells = swiftmap.hexbin(df, resolution=7)
cells.head(3)

In [ ]:
m6 = Map()
m6.add_polygon(cells, color_col="count", name="Counted hexes",
               popup_fields=["h3", "count"], popup_names=["Cell", "Points"])
m6.configure_legend(show=True)
m6

Those are ordinary polygons: a real ramp with numbers, popups on click, and every
targeting and update call that works anywhere else. Counting is the only statistic
built in — sum, mean, or anything else belongs upstream, and whatever upstream
produces paints through this same door.

An H3 cell id is itself a geometry, so an aggregated table needs nothing special:

In [ ]:
m7 = Map()
m7.add_polygon(cells["h3"].iloc[0], name="One cell", color="crimson")
m7